In [1]:
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

c:\Users\swapn\OneDrive\Desktop\new_repo\langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
all_docs=[]
pdf_path = Path("C:\\Users\\swapn\\OneDrive\\Desktop\\new_repo\\langchain\\docs").glob("*.pdf")
for path in pdf_path:
    loader = PyPDFLoader(path)
    docs = loader.load()
    all_docs.extend(docs)

print(f"read {len(all_docs)} documents")

read 48 documents


In [3]:
print(type(all_docs[0]))

<class 'langchain_core.documents.base.Document'>


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks=splitter.split_documents(all_docs)

In [6]:
print(chunks[0].page_content)

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

In [8]:
embedding_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7348.04it/s]


In [9]:
import os
from langchain_community.vectorstores import Chroma

In [10]:
DB_DIR = "./chroma_db"

if os.path.exists(DB_DIR):

    print("Loading existing vector DB...")

    vectorstore = Chroma(
        persist_directory=DB_DIR,
        embedding_function=embedding_model
    )

else:

    print("Creating new vector DB...")

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=DB_DIR
    )

Creating new vector DB...


In [11]:
retriever = vectorstore.as_retriever(
    search_type="mmr", #maximum marginal relevance, it does it based on relevance and diversity of the results
 
    search_kwargs={
        "k": 3,
        "fetch_k": 6
    }
)

In [12]:
query = "numpy"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\nRESULT {i+1}")
    print(doc.page_content[:500])


RESULT 1
in any other way, except for subtracting the mean activity over the training set from each pixel. So
we trained our network on the (centered) raw RGB values of the pixels.
3 The Architecture
The architecture of our network is summarized in Figure 2. It contains eight learned layers —
ﬁve convolutional and three fully-connected. Below, we describe some of the novel or unusual
features of our network’s architecture. Sections 3.1-3.4 are sorted according to our estimation of
their importance, with 

RESULT 2
Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
query with all keys, divide each by√dk, and apply a softmax function to obtain the weights on the
values.
In practice, we compute the attention function on a set of queries simultaneously, packed together
into a matrixQ. The keys and values are also packed together into matricesK andV . We com

In [13]:
from langchain_ollama import ChatOllama

In [14]:
llm = ChatOllama(model="llama3:8b",temperature=0.1)

In [15]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
You are a document question-answering system.

Rules:
- Be concise.
- Do not greet the user.
- Do not say "I'm happy to help".
- Answer ONLY from the provided context.
- If answer is unavailable, say:
  "I could not find the answer in the documents."

Context:
{context}

Question:
{question}
""")

In [16]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

In [17]:
chat_history = []

while True:

    query = input("\nAsk Question: ")

    if query.lower() == "exit":
        break
    retrieved_docs = retriever.invoke(query)
    context = "\n\n".join([
        f"""
    SOURCE: {doc.metadata.get("source")}
    PAGE: {doc.metadata.get("page")}

    CONTENT:
    {doc.page_content}
    """
        for doc in retrieved_docs
    ])
    response = chain.invoke({
        "history": "\n".join(chat_history),
        "context": context,
        "question": query
    })
    print("\nRETRIEVED CHUNKS:\n")

    for i, doc in enumerate(retrieved_docs):
        print(f"\n--- CHUNK {i+1} ---")
        print(doc.page_content[:300])
    print("\nANSWER:")
    print(response)
    chat_history.append(f"User: {query}")
    chat_history.append(f"Assistant: {response}")

    print("\nSOURCES:")

    for i, doc in enumerate(retrieved_docs):

        source = doc.metadata.get("source", "Unknown")

        page = doc.metadata.get("page", "Unknown")

        print(f"{i+1}. {source} | Page {page}")

In [21]:
query = "self attention mechanism"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs):

    print(f"\n--- RESULT {i+1} ---")

    print(doc.metadata)

    print(doc.page_content[:500])


--- RESULT 1 ---
{'created': '2017', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-French translation, we outperform the previoussingle state-of-the-art with model by 0.7 BLEU, achieving a BLEU score of 41.1.', 'creator': 'PyPDF', 'title': 'Attention is All you Need', 'language': 'en-US